In [1]:
import os, sys, shutil, subprocess, zipfile, random, io, warnings
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm
import requests

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
ON_KAGGLE  = os.path.exists("/kaggle")
BASE_DIR   = Path("/kaggle/working") if ON_KAGGLE else Path("./data")
DATA_DIR   = BASE_DIR / "datasets"
CKPT_DIR   = BASE_DIR / "checkpoints"
TU_DIR     = DATA_DIR / "tu_berlin"
SK_DIR     = DATA_DIR / "sketchy"
QD_DIR     = DATA_DIR / "quickdraw"
SKETCH_DIR = DATA_DIR / "sketch_all"
PHOTO_DIR  = DATA_DIR / "photo_all"

for d in [DATA_DIR, CKPT_DIR, TU_DIR, SK_DIR, QD_DIR, SKETCH_DIR, PHOTO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMG_SIZE     = 256
BATCH_SIZE   = 4
NUM_EPOCHS   = 50
LR           = 2e-4
BETAS        = (0.5, 0.999)
LAMBDA_CYCLE = 10.0
LAMBDA_IDT   = 5.0
N_RES        = 9
NGF = NDF    = 64
POOL_SIZE    = 50
NUM_WORKERS  = min(4, os.cpu_count() or 1)

QD_BASE_URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap"
QD_CATS = [
    "airplane", "bicycle", "bird", "boat", "car",
    "cat", "chair", "clock", "dog", "elephant",
    "fish", "flower", "guitar", "horse", "house",
    "shoe", "sun", "tree", "umbrella", "face"
]
QD_MAX_PER = 600

In [3]:
subprocess.run("pip install -q huggingface_hub datasets", shell=True)

CompletedProcess(args='pip install -q huggingface_hub datasets', returncode=0)

In [ ]:
def download_tu_berlin(dst: Path) -> int:
    img_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}
    existing = [p for p in dst.rglob("*") if p.suffix.lower() in img_exts]
    if len(existing) >= 100:
        return len(existing)

    count = 0
    try:
        from datasets import load_dataset
        ds = load_dataset("sdiaeyu6n/tu-berlin", trust_remote_code=True)
        for split_name, split in ds.items():
            img_col = next((c for c in split.column_names
                            if any(k in c.lower() for k in ['image','img','sketch','pixel'])), split.column_names[0])
            for sample in tqdm(split, desc=f"TU-Berlin {split_name}"):
                val = sample[img_col]
                try:
                    if isinstance(val, Image.Image):
                        img = val.convert('RGB')
                    elif isinstance(val, dict) and 'bytes' in val:
                        img = Image.open(io.BytesIO(val['bytes'])).convert('RGB')
                    elif isinstance(val, bytes):
                        img = Image.open(io.BytesIO(val)).convert('RGB')
                    elif isinstance(val, np.ndarray):
                        img = Image.fromarray(val.astype(np.uint8)).convert('RGB')
                    else:
                        continue
                    img.save(dst / f"tu_{count:07d}.png")
                    count += 1
                except Exception:
                    pass
        if count > 0:
            return count
    except Exception:
        pass

    try:
        from huggingface_hub import snapshot_download
        repo_path = snapshot_download(repo_id="sdiaeyu6n/tu-berlin", repo_type="dataset",
                                      local_dir=str(dst / "_raw"))
        for p in Path(repo_path).rglob("*"):
            if p.suffix.lower() in img_exts and p.is_file():
                try:
                    Image.open(p).convert('RGB').save(dst / f"tu_{count:07d}{p.suffix}")
                    count += 1
                except Exception:
                    pass
        shutil.rmtree(dst / "_raw", ignore_errors=True)
    except Exception:
        pass

    return count

download_tu_berlin(TU_DIR)

In [ ]:
def run_kaggle_download(dataset_id: str, dst: Path) -> bool:
    res = subprocess.run(f"kaggle datasets download -d {dataset_id} -p {dst} --unzip",
                         shell=True, capture_output=True, text=True, timeout=600)
    if res.returncode != 0:
        return False
    for zf in dst.rglob("*.zip"):
        try:
            with zipfile.ZipFile(zf) as z: z.extractall(zf.parent)
            zf.unlink()
        except Exception:
            pass
    return True


def find_sketch_photo_dirs(root: Path):
    img_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.webp', '.gif'}
    sk_dirs, ph_dirs = [], []

    def has_images(d):
        return any(p.suffix.lower() in img_exts for p in d.iterdir() if p.is_file())

    for dirpath, _, _ in os.walk(root):
        p = Path(dirpath)
        if not p.is_dir() or not any(p.iterdir()): continue
        try:
            if not has_images(p): continue
        except Exception:
            continue
        nm = p.name.lower()
        parent_nm = p.parent.name.lower()
        if any(k in nm for k in ['sketch', 'drawing', 'svg', 'line']):
            sk_dirs.append(p)
        elif any(k in nm for k in ['photo', 'image', 'img', 'pic', 'rgb', 'ref', 'real']):
            ph_dirs.append(p)
        elif any(k in parent_nm for k in ['sketch', 'drawing']):
            sk_dirs.append(p)
        elif any(k in parent_nm for k in ['photo', 'image', 'img']):
            ph_dirs.append(p)
    return sk_dirs, ph_dirs


def download_sketchy(sk_raw: Path, sketch_dst: Path, photo_dst: Path):
    img_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

    for did in ["prasunroy/sketchy", "datasciencemaster/sketchy", "sharanyasundar/sketchy-dataset"]:
        ok = run_kaggle_download(did, sk_raw)
        if ok and sum(1 for p in sk_raw.rglob("*") if p.suffix.lower() in img_exts) >= 50:
            break

    sk_dirs, ph_dirs = find_sketch_photo_dirs(sk_raw)
    n_sk = n_ph = 0

    for d in tqdm(sk_dirs, desc="Sketchy sketches"):
        for p in d.glob("*"):
            if p.is_file() and p.suffix.lower() in img_exts:
                try:
                    Image.open(p).convert('RGB').save(sketch_dst / f"sk_{n_sk:07d}.png")
                    n_sk += 1
                except Exception:
                    pass

    for d in tqdm(ph_dirs, desc="Sketchy photos"):
        for p in d.glob("*"):
            if p.is_file() and p.suffix.lower() in img_exts:
                try:
                    Image.open(p).convert('RGB').save(photo_dst / f"skph_{n_ph:07d}.png")
                    n_ph += 1
                except Exception:
                    pass

download_sketchy(SK_DIR, SKETCH_DIR, PHOTO_DIR)

In [ ]:
def download_quickdraw(dst: Path, cats: list, max_per=600, size=256):
    for cat in tqdm(cats, desc="QuickDraw"):
        if len(list(dst.glob(f"qd_{cat}_*.png"))) >= max_per:
            continue
        npy_path = dst / f"_{cat}.npy"
        try:
            r = requests.get(f"{QD_BASE_URL}/{cat}.npy", stream=True, timeout=120)
            r.raise_for_status()
            with open(npy_path, 'wb') as f:
                for chunk in r.iter_content(32768): f.write(chunk)
            data = np.load(npy_path, allow_pickle=False)
            npy_path.unlink(missing_ok=True)
            for i in range(min(max_per, len(data))):
                arr = data[i].reshape(28, 28).astype(np.uint8)
                img = Image.fromarray(arr, mode='L').resize((size, size), Image.BILINEAR).convert('RGB')
                img.save(dst / f"qd_{cat}_{i:05d}.png")
        except Exception:
            npy_path.unlink(missing_ok=True)

download_quickdraw(SKETCH_DIR, QD_CATS, QD_MAX_PER, IMG_SIZE)

In [ ]:
def download_coco_val(dst: Path, max_photos=5000) -> int:
    existing = list(dst.glob("coco_*.png"))
    if len(existing) >= max_photos:
        return len(existing)

    coco_zip = dst.parent / "_coco_val.zip"
    coco_raw = dst.parent / "_coco_raw"
    coco_raw.mkdir(exist_ok=True)

    if not coco_zip.exists():
        r = requests.get("http://images.cocodataset.org/zips/val2017.zip",
                         stream=True, timeout=600)
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        bar = tqdm(total=total, unit='B', unit_scale=True, desc="COCO val2017")
        with open(coco_zip, 'wb') as f:
            for chunk in r.iter_content(65536):
                f.write(chunk); bar.update(len(chunk))
        bar.close()

    with zipfile.ZipFile(coco_zip) as z:
        z.extractall(coco_raw)
    coco_zip.unlink(missing_ok=True)

    count = 0
    for p in tqdm(sorted(coco_raw.rglob("*.jpg"))[:max_photos], desc="Saving COCO"):
        try:
            Image.open(p).convert('RGB').save(dst / f"coco_{count:06d}.png")
            count += 1
        except Exception:
            pass

    shutil.rmtree(coco_raw, ignore_errors=True)
    return count

n_ph_now = len(list(PHOTO_DIR.glob("*.png")))
if n_ph_now < 2000:
    download_coco_val(PHOTO_DIR, max_photos=5000)

In [ ]:
class SketchPhotoDataset(Dataset):
    def __init__(self, sketch_dir, photo_dir, img_size=256, is_train=True):
        exts = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}
        self.sketches = sorted(p for p in Path(sketch_dir).rglob("*")
                               if p.is_file() and p.suffix.lower() in exts)
        self.photos   = sorted(p for p in Path(photo_dir).rglob("*")
                               if p.is_file() and p.suffix.lower() in exts)
        load_sz = img_size + 30

        self.sk_tf = T.Compose([
            T.Resize(load_sz, T.InterpolationMode.BILINEAR),
            T.RandomCrop(img_size) if is_train else T.CenterCrop(img_size),
            T.RandomHorizontalFlip(0.5) if is_train else T.Lambda(lambda x: x),
            T.RandomRotation(10)        if is_train else T.Lambda(lambda x: x),
            T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3),
        ])
        self.ph_tf = T.Compose([
            T.Resize(load_sz, T.InterpolationMode.BICUBIC),
            T.RandomCrop(img_size) if is_train else T.CenterCrop(img_size),
            T.RandomHorizontalFlip(0.5) if is_train else T.Lambda(lambda x: x),
            T.ColorJitter(0.2, 0.2, 0.1, 0.05) if is_train else T.Lambda(lambda x: x),
            T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3),
        ])

    def __len__(self): return max(len(self.sketches), len(self.photos))

    def __getitem__(self, idx):
        sp = self.sketches[idx % len(self.sketches)]
        pp = self.photos[idx   % len(self.photos)]
        try: sk = Image.open(sp).convert('RGB')
        except: sk = Image.open(self.sketches[0]).convert('RGB')
        try: ph = Image.open(pp).convert('RGB')
        except: ph = Image.open(self.photos[0]).convert('RGB')
        return {'A': self.sk_tf(sk), 'B': self.ph_tf(ph)}

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim), nn.ReLU(True),
            nn.ReflectionPad2d(1), nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim))
    def forward(self, x): return x + self.block(x)


class Generator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ngf=64, n_res=9):
        super().__init__()
        m = [nn.ReflectionPad2d(3), nn.Conv2d(in_ch, ngf, 7),
             nn.InstanceNorm2d(ngf), nn.ReLU(True)]
        for i in range(2):
            f = 2**i
            m += [nn.Conv2d(ngf*f, ngf*f*2, 3, 2, 1), nn.InstanceNorm2d(ngf*f*2), nn.ReLU(True)]
        for _ in range(n_res): m.append(ResBlock(ngf*4))
        for i in range(2, 0, -1):
            f = 2**i
            m += [nn.ConvTranspose2d(ngf*f, ngf*f//2, 3, 2, 1, 1),
                  nn.InstanceNorm2d(ngf*f//2), nn.ReLU(True)]
        m += [nn.ReflectionPad2d(3), nn.Conv2d(ngf, out_ch, 7), nn.Tanh()]
        self.model = nn.Sequential(*m)
    def forward(self, x): return self.model(x)


class PatchDisc(nn.Module):
    def __init__(self, in_ch=3, ndf=64):
        super().__init__()
        def blk(i, o, norm=True, s=2):
            layers = [nn.Conv2d(i, o, 4, s, 1)]
            if norm: layers.append(nn.InstanceNorm2d(o))
            return layers + [nn.LeakyReLU(0.2, True)]
        self.model = nn.Sequential(
            *blk(in_ch, ndf, norm=False),
            *blk(ndf,   ndf*2),
            *blk(ndf*2, ndf*4),
            *blk(ndf*4, ndf*8, s=1),
            nn.Conv2d(ndf*8, 1, 4, 1, 1))
    def forward(self, x): return self.model(x)


def init_w(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.InstanceNorm2d) and m.weight is not None:
        nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

G_AB = Generator(3, 3, NGF, N_RES).to(device); G_AB.apply(init_w)
G_BA = Generator(3, 3, NGF, N_RES).to(device); G_BA.apply(init_w)
D_A  = PatchDisc(3, NDF).to(device);            D_A.apply(init_w)
D_B  = PatchDisc(3, NDF).to(device);            D_B.apply(init_w)

In [ ]:
class LSGANLoss(nn.Module):
    mse = nn.MSELoss()
    def forward(self, pred, real: bool):
        t = torch.ones_like(pred) if real else torch.zeros_like(pred)
        return self.mse(pred, t)


class ImagePool:
    def __init__(self, size=50):
        self.size = size; self.pool = []

    def query(self, images: torch.Tensor) -> torch.Tensor:
        out = []
        for img in images.unbind(0):
            img = img.unsqueeze(0)
            if len(self.pool) < self.size:
                self.pool.append(img); out.append(img)
            elif random.random() > 0.5:
                idx = random.randrange(self.size)
                old = self.pool[idx].clone()
                self.pool[idx] = img; out.append(old)
            else:
                out.append(img)
        return torch.cat(out, 0)


gan_crit = LSGANLoss().to(device)
cyc_crit = nn.L1Loss()
idt_crit = nn.L1Loss()
pool_A   = ImagePool(POOL_SIZE)
pool_B   = ImagePool(POOL_SIZE)

In [ ]:
opt_G  = optim.Adam(list(G_AB.parameters()) + list(G_BA.parameters()), lr=LR, betas=BETAS)
opt_DA = optim.Adam(D_A.parameters(), lr=LR, betas=BETAS)
opt_DB = optim.Adam(D_B.parameters(), lr=LR, betas=BETAS)

def lr_lambda(epoch):
    decay_start = NUM_EPOCHS // 2
    return 1.0 if epoch < decay_start else max(0.0, 1 - (epoch - decay_start) / (NUM_EPOCHS - decay_start))

sched_G  = optim.lr_scheduler.LambdaLR(opt_G,  lr_lambda)
sched_DA = optim.lr_scheduler.LambdaLR(opt_DA, lr_lambda)
sched_DB = optim.lr_scheduler.LambdaLR(opt_DB, lr_lambda)
scaler   = GradScaler()

state = dict(epoch=0, G=[], DA=[], DB=[], cyc=[], idt=[], best_cyc=float('inf'))